<a href="https://colab.research.google.com/github/gez2code/dermamnist-hybrid-study/blob/main/Binary_Classification_DermaMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# BLOCK 1: INSTALLATION & IMPORTS
# ============================================================================
!pip install medmnist wandb

import os
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Deep Learning Imports
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, VGG16, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K

# Metrics & Data
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
from sklearn.utils import class_weight
from medmnist import DermaMNIST

# Tracking
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

# Reproducibility
SEED = 42
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds()
print(f"✓ Setup complete. TensorFlow Version: {tf.__version__}")

In [ ]:
# ============================================================================
# BLOCK 2: DATA LOADING & PREPROCESSING
# ============================================================================
def load_and_preprocess_data():
    print("\nLoading DermaMNIST...")
    train_data = DermaMNIST(split='train', download=True, size=28)
    val_data = DermaMNIST(split='val', download=True, size=28)
    test_data = DermaMNIST(split='test', download=True, size=28)

    # Normalize
    x_train = train_data.imgs.astype('float32') / 255.0
    x_val = val_data.imgs.astype('float32') / 255.0
    x_test = test_data.imgs.astype('float32') / 255.0

    # Binary mapping: Malignant (1) vs Benign (0)
    to_binary = lambda y: np.isin(y, [0, 1, 6]).astype(int)
    y_train_bin = to_binary(train_data.labels)
    y_val_bin = to_binary(val_data.labels)
    y_test_bin = to_binary(test_data.labels)

    # One-hot encode
    y_train = tf.keras.utils.to_categorical(y_train_bin, 2)
    y_val = tf.keras.utils.to_categorical(y_val_bin, 2)
    y_test = tf.keras.utils.to_categorical(y_test_bin, 2)

    # --- FIX: Dampened Class Weights (1:3 ratio instead of 1:9) ---
    # Prevents the model from just guessing "Malignant" for everything.
    class_weights = {0: 1.0, 1: 3.0}
    print(f'✓ Data loaded. Weights configured: {class_weights}')

    return {
        'x_train': x_train, 'y_train': y_train, 'y_train_bin': y_train_bin,
        'x_val': x_val, 'y_val': y_val, 'y_val_bin': y_val_bin,
        'x_test': x_test, 'y_test': y_test, 'y_test_bin': y_test_bin,
        'class_weights': class_weights
    }

def get_augmentation_generator():
    return ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        zoom_range=0.1,
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='nearest'
    )

# Load Data Once
data = load_and_preprocess_data()
datagen = get_augmentation_generator()

# ============================================================================
# GOOGLE DRIVE VERBINDEN & SPEICHER-FUNKTION
# ============================================================================
from google.colab import drive
import shutil
import os

# 1. Drive mounten
drive.mount('/content/drive')

# 2. Ordner erstellen
SAVE_DIR = '/content/drive/MyDrive/DermaModels_Phase1'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Speicherort: {SAVE_DIR}")

# 3. Hilfsfunktion zum Speichern
def save_to_drive(experiment_name):
    source = f"{experiment_name}.keras"
    destination = os.path.join(SAVE_DIR, source)

    if os.path.exists(source):
        shutil.copy(source, destination)
        print(f"💾 GESPEICHERT: {experiment_name} -> Google Drive")
    else:
        print(f"⚠️ FEHLER: Modelldatei {source} nicht gefunden!")

In [ ]:
# ============================================================================
# BLOCK 3: MODEL ARCHITECTURES
# ============================================================================
def build_custom_cnn(filters_base=32, depth=3, dropout=0.5, dense_units=512):
    model = models.Sequential([layers.Input(shape=(28, 28, 3))])
    for i in range(depth):
        filters = filters_base * (2 ** i)
        model.add(layers.Conv2D(filters, (3, 3), activation='relu', padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))
        model.add(layers.Dropout(dropout * (0.5 + i*0.25)))

    model.add(layers.Flatten())
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(2, activation='softmax'))
    return model

def build_transfer_model(base_name='resnet50', dropout=0.5, unfreeze_layers=None, dense_units=256):
    inputs = layers.Input(shape=(28, 28, 3))
    # Upsampling is crucial for pre-trained models
    x = layers.UpSampling2D(size=(2, 2), interpolation='bilinear')(inputs)

    base_models = {
        'resnet50': lambda: ResNet50(include_top=False, weights='imagenet', input_shape=(56, 56, 3), pooling='avg'),
        'vgg16': lambda: VGG16(include_top=False, weights='imagenet', input_shape=(56, 56, 3), pooling='avg'),
        'efficientnet': lambda: EfficientNetB0(include_top=False, weights='imagenet', input_shape=(56, 56, 3), pooling='avg')
    }

    if base_name not in base_models: raise ValueError(f"Unknown base: {base_name}")
    base = base_models[base_name]()

    # Fine-tuning logic
    if unfreeze_layers is None:
        base.trainable = True # Full fine-tuning
    elif unfreeze_layers == 0:
        base.trainable = False # Feature extraction
    else:
        base.trainable = True
        for layer in base.layers[:-unfreeze_layers]:
            layer.trainable = False

    x = base(x, training=False)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout * 0.5)(x)
    outputs = layers.Dense(2, activation='softmax')(x)

    return models.Model(inputs, outputs, name=f"{base_name}_model")

In [ ]:
# ============================================================================
# BLOCK 4: TRAINING CONFIGURATION & HELPERS
# ============================================================================
def compile_model(model, learning_rate):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            # class_id=1 ensures we measure Malignant performance, not global accuracy
            tf.keras.metrics.Precision(class_id=1, name='precision'),
            tf.keras.metrics.Recall(class_id=1, name='recall')
        ]
    )

def get_callbacks(config, model_path):
    return [
        # FIX: Monitor AUC to save the "smartest" model, not the one guessing "1" everywhere
        callbacks.EarlyStopping(monitor='val_auc', patience=config['patience'], mode='max', restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(filepath=model_path, monitor='val_auc', mode='max', save_best_only=True, verbose=0),

        # W&B Logging
        WandbMetricsLogger(log_freq='epoch'),
        WandbModelCheckpoint(model_path, monitor='val_auc', mode='max', save_best_only=True)
    ]

def compute_metrics(model, data_dict, split_name):
    """Calculates comprehensive metrics for a specific split"""
    y_pred_probs = model.predict(data_dict['x'], verbose=0)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    y_true_classes = np.argmax(data_dict['y'], axis=1)

    metrics = {
        f'{split_name}/accuracy': accuracy_score(y_true_classes, y_pred_classes),
        f'{split_name}/auc': roc_auc_score(data_dict['y'], y_pred_probs),
        f'{split_name}/recall_mal': recall_score(y_true_classes, y_pred_classes, pos_label=1, zero_division=0),
        f'{split_name}/precision_mal': precision_score(y_true_classes, y_pred_classes, pos_label=1, zero_division=0),
        f'{split_name}/f1_mal': f1_score(y_true_classes, y_pred_classes, pos_label=1, zero_division=0)
    }
    return metrics, confusion_matrix(y_true_classes, y_pred_classes)

In [ ]:
# ============================================================================
# BLOCK 5: MAIN TRAINING LOOP
# ============================================================================
def train_experiment(config, data, datagen):
    print(f"\n{'='*60}")
    print(f"🚀 STARTING: {config['name']}")
    print(f"{'='*60}")

    # 1. Init W&B (Generate ID to prevent 409 errors)
    if wandb.run is not None: wandb.finish()
    run = wandb.init(project="DermaMNIST_Study", name=config['name'], config=config, reinit=True, id=wandb.util.generate_id())

    # 2. Build Model
    if config['architecture'] == 'custom_cnn':
        model = build_custom_cnn(config['filters_base'], config['depth'], config['dropout'], config['dense_units'])
    else:
        model = build_transfer_model(config['architecture'], config['dropout'], config['unfreeze_layers'], config['dense_units'])

    compile_model(model, config['learning_rate'])
    model_path = f"{config['name']}.keras"

    # 3. Train
    try:
        history = model.fit(
            datagen.flow(data['x_train'], data['y_train'], batch_size=config['batch_size'], seed=SEED),
            steps_per_epoch=len(data['x_train']) // config['batch_size'],
            epochs=config['epochs'],
            validation_data=(data['x_val'], data['y_val']),
            class_weight=data['class_weights'],
            callbacks=get_callbacks(config, model_path),
            verbose=1
        )

        # 4. Evaluation (Train/Val/Test)
        train_metrics, _ = compute_metrics(model, {'x': data['x_train'], 'y': data['y_train']}, 'train')
        val_metrics, _ = compute_metrics(model, {'x': data['x_val'], 'y': data['y_val']}, 'val')
        test_metrics, cm = compute_metrics(model, {'x': data['x_test'], 'y': data['y_test']}, 'test')

        # 5. Log Results
        all_metrics = {**train_metrics, **val_metrics, **test_metrics}
        wandb.log(all_metrics)
        print(f"\n🏁 RESULT: Val AUC: {val_metrics['val/auc']:.4f} | Test Recall: {test_metrics['test/recall_mal']:.4f}")
        print(f"Confusion Matrix (Test):\n{cm}")

    except Exception as e:
        print(f"❌ Error in experiment {config['name']}: {e}")
        all_metrics = {}

    finally:
        # 6. CRITICAL: Cleanup Memory
        wandb.finish()
        K.clear_session()
        del model
        gc.collect()

    return {**all_metrics, 'config': config}

In [ ]:
# ============================================================================
# BLOCK 6a: Custom CNN (Trainieren & Speichern)
# ============================================================================
config_cnn = {
    'name': 'P1_CNN_Baseline',
    'architecture': 'custom_cnn',
    'filters_base': 32, 'depth': 3,
    'dropout': 0.5, 'dense_units': 512,
    'learning_rate': 0.001, 'batch_size': 64,
    'epochs': 30,
    'patience': 10,
    'unfreeze_layers': None
}

# 1. Trainieren
res_cnn = train_experiment(config_cnn, data, datagen)

# 2. Sofort ins Drive sichern
save_to_drive(config_cnn['name'])

In [ ]:
# ============================================================================
# BLOCK 6b: ResNet50 (Trainieren & Speichern)
# ============================================================================
config_resnet = {
    'name': 'P1_ResNet50_Full',
    'architecture': 'resnet50',
    'dropout': 0.5, 'dense_units': 256, 'unfreeze_layers': None,
    'learning_rate': 0.0001, 'batch_size': 32, 'epochs': 30, 'patience': 10
}

res_resnet = train_experiment(config_resnet, data, datagen)
save_to_drive(config_resnet['name'])

In [ ]:
# ============================================================================
# BLOCK 6c: VGG16 (Trainieren & Speichern)
# ============================================================================
config_vgg = {
    'name': 'P1_VGG16_Full',
    'architecture': 'vgg16',
    'dropout': 0.5, 'dense_units': 256, 'unfreeze_layers': None,
    'learning_rate': 0.0001, 'batch_size': 32, 'epochs': 30, 'patience': 10
}

res_vgg = train_experiment(config_vgg, data, datagen)
save_to_drive(config_vgg['name'])

In [ ]:
# ============================================================================
# BLOCK 6d: EfficientNet (Trainieren & Speichern)
# ============================================================================
config_effnet = {
    'name': 'P1_EfficientNet_Full',
    'architecture': 'efficientnet',
    'dropout': 0.5, 'dense_units': 256, 'unfreeze_layers': None,
    'learning_rate': 0.0001, 'batch_size': 32, 'epochs': 30, 'patience': 10
}

res_effnet = train_experiment(config_effnet, data, datagen)
save_to_drive(config_effnet['name'])

In [ ]:
# ============================================================================
# BLOCK 6e: COMPARE RESULTS
# ============================================================================

# Sammle alle verfügbaren Ergebnisse
all_results = []

# Prüfen, welche Variablen existieren (falls du eine Zelle übersprungen hast)
if 'res_cnn' in locals(): all_results.append(res_cnn)
if 'res_resnet' in locals(): all_results.append(res_resnet)
if 'res_vgg' in locals(): all_results.append(res_vgg)
if 'res_effnet' in locals(): all_results.append(res_effnet)

if all_results:
    df_compare = pd.DataFrame(all_results)

    # Relevante Spalten auswählen
    display_cols = [
        'config',
        'val/auc', 'val/recall_mal',
        'test/auc', 'test/recall_mal', 'test/precision_mal'
    ]

    # Config Dictionary zu String vereinfachen für die Anzeige
    df_compare['Architecture'] = df_compare['config'].apply(lambda x: x['architecture'])

    print("\n🏆 PHASE 1 RESULTS SUMMARY:")
    print(df_compare[['Architecture', 'val/auc', 'val/recall_mal', 'test/auc', 'test/recall_mal']])
else:
    print("Noch keine Ergebnisse verfügbar. Bitte führe Blöcke 6a-6d aus.")

In [ ]:
# ============================================================================
# BLOCK 7c: MODELL LADEN & WEITERVERWENDEN
# ============================================================================

# 1. Welches Modell möchtest du laden? (Name aus der Config oben)
MODEL_NAME_TO_LOAD = f'P2_{BASE_ARCH}_HighDropout.keras'
path_to_model = os.path.join(SAVE_DIR, MODEL_NAME_TO_LOAD)

print(f"Lade Modell von: {path_to_model} ...")

try:
    # Modell laden
    loaded_model = tf.keras.models.load_model(path_to_model)
    print("✅ Modell erfolgreich geladen!")

    # Optional: Schnelltest ob es funktioniert
    print("Führe Test-Prädiktion durch...")
    test_metrics, _ = compute_metrics(loaded_model, {'x': data['x_test'], 'y': data['y_test']}, 'loaded_test')
    print(f"Test AUC des geladenen Modells: {test_metrics['loaded_test/auc']:.4f}")

    # Hier könntest du jetzt Phase 3 (Feintuning) mit diesem Modell starten
    # z.B. loaded_model.fit(...) mit kleinerer Learning Rate

except OSError:
    print(f"❌ Fehler: Datei nicht gefunden. Hast du Block 7a/7b ausgeführt?")
except Exception as e:
    print(f"❌ Fehler beim Laden: {e}")

In [ ]:
# ============================================================================
# BLOCK 7d: PHASE 3 - FINE-TUNING & SPEICHERN
# ============================================================================

# 1. Konfiguration für Phase 3 (Fine-Tuning)
# Wir nehmen das geladene Modell und trainieren es sanft weiter
finetune_config = {
    'name': f'P3_{BASE_ARCH}_Final_Finetuned',  # Neuer Name!
    'learning_rate': 1e-5,  # Sehr kleine Rate für Fine-Tuning
    'epochs': 20,           # Weniger Epochen reichen oft
    'batch_size': 32
}

print(f"🚀 Starte Fine-Tuning für: {finetune_config['name']}")

# 2. Modell neu kompilieren (wichtig bei Änderung der Learning Rate!)
# Wir nutzen den gleichen Optimizer, aber mit kleinerer Rate
loaded_model.compile(
    optimizer=optimizers.Adam(learning_rate=finetune_config['learning_rate']),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(class_id=1, name='precision'),
        tf.keras.metrics.Recall(class_id=1, name='recall')
    ]
)

# 3. Training starten (mit dem geladenen Modell)
# Hinweis: Wir nutzen 'loaded_model' direkt weiter
history_finetune = loaded_model.fit(
    datagen.flow(data['x_train'], data['y_train'], batch_size=finetune_config['batch_size'], seed=SEED),
    steps_per_epoch=len(data['x_train']) // finetune_config['batch_size'],
    epochs=finetune_config['epochs'],
    validation_data=(data['x_val'], data['y_val']),
    class_weight=data['class_weights'],
    # Wir nutzen wieder Callbacks, aber speichern unter dem NEUEN Namen
    callbacks=get_callbacks({'patience': 5}, f"{finetune_config['name']}.keras"),
    verbose=1
)

# 4. DAS WICHTIGSTE: Das neue Modell speichern!
save_to_drive(finetune_config['name'])

print("\n✅ Phase 3 abgeschlossen und Modell im Drive gesichert.")